# 实验六 · 直方图统计 —— 数据竞争与伪共享

**所属**：《并行计算技术》第五章 · OpenMP 编程　|　**难度**：⭐⭐⭐⭐ 难点　|　**预计时长**：30–40 分钟

实验五讨论了并行区域本身的结构性开销。本实验转而讨论**数据访问**层面的开销：当多个线程必须更新同一批数据时，性能代价的主要来源是什么。七个版本沿两条主线展开：其一是四种同步手段的代价对比，其二是一种无需任何同步、却仍会显著影响性能的隐性开销——伪共享。

> **实验说明**
> 1. 本实验采用**递进式的版本组织**：以串行实现为基准，每个版本仅引入一种新的 OpenMP 构造或一处相应的代码改写，并在同一次运行中完成全部版本的计时与正确性校验，因而各版本面对的是完全相同的数据与运行环境，各版本之间具有可比性。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖支持 OpenMP 的 **GCC 编译器**，建议在华为鲲鹏处理器或其他 AArch64 平台上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 本实验需要运行**两次**，分别取 `num_bins = 100` 与 `num_bins = 8`。后者会使每个线程的私有区域缩小到 32 字节，多个线程的私有区域因而落入同一条缓存行，伪共享的影响随之放大。
> 6. 本实验的直方图**写入地址由数据本身决定**，无法像求和那样直接套用标量 `reduction`，因而适合作为讨论多种同步手段的案例。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明直方图统计为何比求和更难并行：写入地址由数据决定，事先无法确定
- 掌握四种互斥手段的语法与代价：`critical`、`atomic`、`omp_lock_t` 数组、以及线程局部化
- 理解 `atomic` 与 `critical` 的区别：前者由单条硬件读改写指令支撑，后者需要完整的锁协议
- 解释**缓存一致性协议**下写共享行的代价，并说明为何「无锁」不等于「无开销」
- 给出**伪共享**（false sharing）的定义，能够根据数据布局判断一段代码是否存在伪共享
- 掌握**缓存行填充**（padding）的实施方法，并计算给定 bin 数下所需的填充量
- 使用 OpenMP 4.5 的**数组段归约** `reduction(+ : a[:n])`，并用 `_OPENMP` 宏做能力检测

## 🗺️ 学习路径

1. **准备阶段**：理解直方图的写入模式与竞争成因
2. **V0 串行基准** → **V1 `critical`**：全局互斥，最简单也最慢
3. **V2 `atomic`**：以硬件原子指令替代锁协议
4. **V3 锁数组**：细化锁的粒度，每个 bin 一把锁
5. **V4 线程局部直方图**：彻底消除竞争，但引入伪共享
6. **V5 缓存行填充**：消除伪共享
7. **V6 数组段归约**：由编译器完成局部化与合并
8. **双规模对照**：以 100 与 8 两种 bin 数运行，观察伪共享强度的变化
9. **分析与扩展**：bin 数扫描，定位伪共享的作用区间

## 1. 背景知识：直方图统计

### 1.1 问题描述

给定一个包含 $n$ 个整数的数组，每个元素的取值范围为 $[0, B)$，统计每个取值出现的次数：

```c
for (long i = 0; i < n; i++) {
  bins[data[i]]++;
}
```

这段代码虽仅有一行，却对应并行编程中一类较难处理的模式。

### 1.2 为何比求和更难

对照实验二的求和：

| | 求和 `sum += x[i]` | 直方图 `bins[data[i]]++` |
|---|---|---|
| 写入目标 | **固定**的一个标量 | **由数据决定**的某个数组元素 |
| 冲突判断 | 必然冲突 | 取决于运行时数据 |
| 归约方式 | 标量归约，编译器直接支持 | 需要整个数组的归约 |

求和的写入目标在编译期即可确定，`reduction(+ : sum)` 可直接处理。直方图的写入目标 `bins[data[i]]` 需在运行时根据 `data[i]` 确定，编译器难以为其生成简单的私有累加。

这类模式称为**不规则归约**（irregular reduction），在图算法、粒子模拟、稀疏矩阵组装中都很常见。

### 1.3 竞争的具体形态

`bins[b]++` 展开后是三条指令：

```text
  Load   R1 ← bins[b]
  Add    R1 ← R1 + 1
  Store  bins[b] ← R1
```

两个线程若同时对同一个 bin 执行这三条指令，可能出现如下交错：

```text
  线程 A: Load(5) ────────────── Add→6 ── Store(6)
  线程 B:        Load(5) ── Add→6 ── Store(6)
                                              ↑
                          两次自增，结果却只增加了 1
```

这与第四章讨论的丢失更新属于同一类问题。冲突概率取决于 bin 数：bin 越少，多个线程访问同一 bin 的概率越高。

## 2. 竞争的管理与消除

### 2.1 `critical` 与 `atomic`

```c
#pragma omp critical      // 保护任意结构化块
{ bins[b]++; }

#pragma omp atomic        // 只能保护单条读改写语句
bins[b]++;
```

| | `critical` | `atomic` |
|---|---|---|
| 保护范围 | 任意结构化块 | 单条读改写语句 |
| 实现方式 | 锁协议（获取、释放） | 硬件原子指令 |
| 互斥粒度 | **全局**：所有无名 `critical` 共用一把锁 | **按地址**：不同地址互不影响 |
| 典型代价 | 高 | 中 |

**`critical` 的全局互斥特性是其在本实验中性能最差的主要原因**：即便两个线程访问的是完全不同的 bin，仍需通过同一把锁串行化。

`atomic` 在**语义**上以存储位置为单位：规范只保证对同一地址的读改写不被打断，访问不同 bin 的线程之间并无互斥关系。但在**实现**上，AArch64 的 `LDXR` / `STXR`（Load-Exclusive / Store-Exclusive）配对所使用的独占预留粒度（exclusive reservation granule）通常等于一条缓存行，因此写入同一缓存行的不同地址仍会相互干扰——这正是 3.1 节所述伪共享的另一种表现。

`atomic` 支持的形式包括：

```c
#pragma omp atomic update    // x++, x += e （默认形式）
#pragma omp atomic read      // v = x
#pragma omp atomic write     // x = e
#pragma omp atomic capture   // v = x++ ，同时读出旧值并更新
```

### 2.2 锁数组：细化互斥粒度

```c
omp_lock_t *locks = malloc(num_bins * sizeof(omp_lock_t));
for (int b = 0; b < num_bins; b++) omp_init_lock(&locks[b]);
// ...
omp_set_lock(&locks[b]);
bins[b]++;
omp_unset_lock(&locks[b]);
// ...
for (int b = 0; b < num_bins; b++) omp_destroy_lock(&locks[b]);
```

其思路是：`critical` 的问题在于互斥粒度过粗，因此为每个 bin 单独配置一把锁，两个线程访问不同 bin 时便无需互相等待。

**但该方案通常效率较低**，原因有二：

1. `omp_lock_t` 本身是一个需要频繁读写的共享对象，锁数组自身成为新的竞争热点；
2. 加锁与解锁两次函数调用的代价，高于 `atomic` 的单条原子指令。

### 2.3 线程局部化：消除竞争而非管理竞争

前三种方案都在**管理**竞争。第四种方案直接**消除**它：

```c
int *local_bins = alloc(thread_count * num_bins * sizeof(int));

#pragma omp parallel
{
  int tid = omp_get_thread_num();
  int *my_bins = local_bins + tid * num_bins;   // 各线程互不重叠
#pragma omp for
  for (long i = 0; i < n; i++) my_bins[data[i]]++;   // 无需任何同步
}

for (int t = 0; t < thread_count; t++)             // 串行合并
  for (int b = 0; b < num_bins; b++)
    bins[b] += local_bins[t * num_bins + b];
```

每个线程写入自己那一段，地址互不重叠，因而不存在数据竞争。**但性能仍可能不佳**，原因见第 3 节。

## 3. 伪共享与数组段归约

### 3.1 伪共享

**缓存一致性的基本单位是缓存行**，而不是字节。在多数 Arm 与 x86 处理器上，缓存行长度为 64 字节。

当两个线程写入**不同的变量**，但这两个变量恰好位于**同一条缓存行**时，缓存一致性协议仍会把该行在两个核心之间反复迁移。程序逻辑上并不存在共享，但硬件层面仍因缓存一致性协议而付出了共享的代价，这一现象称为**伪共享**（false sharing）。

```text
  未填充（num_bins = 8，每线程 32 字节）：

  ┌────────── 缓存行 0（64 字节）──────────┐┌── 缓存行 1 ──┐
  │ 线程0 的 8 个 bin │ 线程1 的 8 个 bin  ││ 线程2 ...     │
  └───────────────────────────────────────┘└──────────────┘
        ↑ 线程 0 与线程 1 写同一条行，该行在两核之间来回迁移

  填充后（每线程独占整数条缓存行）：

  ┌── 缓存行 0 ──┐┌── 缓存行 1 ──┐┌── 缓存行 2 ──┐
  │ 线程0 + 填充 ││ 线程1 + 填充 ││ 线程2 + 填充 │
  └──────────────┘└──────────────┘└──────────────┘
        ↑ 各线程独占各自的行，无迁移
```

**判断伪共享的三个条件**（需同时满足）：

1. 多个线程频繁**写入**；
2. 写入的地址**不同**（若相同则是真共享）；
3. 这些地址落在**同一条缓存行**内。

### 3.2 缓存行填充

消除伪共享的方法是把每个线程的私有区域**向上取整到整数条缓存行**：

```c
size_t bytes_per_thread = num_bins * sizeof(int);
size_t padded_bytes =
    ((bytes_per_thread + CACHE_LINE - 1) / CACHE_LINE) * CACHE_LINE;
size_t stride = padded_bytes / sizeof(int);
int *my_bins = local_bins + tid * stride;   // 按填充后的步长索引
```

以 `num_bins = 8` 为例：原始占用 32 字节，填充后为 64 字节，内存开销翻倍，但每个线程独占一条完整的缓存行。

> **仅有填充还不够**：`local_bins` 的**起始地址**也必须对齐到缓存行边界，否则第一个线程的区域会跨越两条行，填充的效果被破坏。本实验统一使用 `alloc_aligned()` 包装 `aligned_alloc`，保证 64 字节对齐。这与第四章伪共享实验中讨论的「栈变量用 `_Alignas`、堆内存必须用 `aligned_alloc`」是同一要点。

### 3.3 数组段归约（OpenMP 4.5）

```c
#pragma omp parallel for reduction(+ : bins[ : num_bins])
for (long i = 0; i < n; i++) {
  bins[data[i]]++;
}
```

语法 `bins[start : length]` 表示对数组的一个连续区段做归约。编译器会自动完成 V4/V5 中手工编写的全部工作：分配私有副本、以 0 初始化、循环结束后合并。填充与否由实现决定，通常也会考虑对齐。

该特性自 OpenMP 4.5 起可用，因此需要能力检测：

```c
#if _OPENMP >= 201511
#define HAVE_ARRAY_REDUCTION 1
#else
#define HAVE_ARRAY_REDUCTION 0
#endif
```

这正是实验一 3.4 节所述 `_OPENMP` 宏第二种用法的实例。

## 4. 环境准备与检查

本节确认三项内容：编译器是否支持 OpenMP、运行时报告的处理器数量、以及各处理器核心的最大频率是否一致。

第三项检查针对**异构多核**平台。Arm 的 big.LITTLE 架构把高性能核心与高能效核心集成在同一块芯片上，二者的频率与微架构均不相同。在这类平台上，同一段代码在不同类型的核心上执行，耗时可能相差 2 倍以上，线程数与加速比之间因而不再是简单的线性关系。

需要说明的是，最大频率不一致只是异构多核的**必要非充分**证据：同构多核平台也可能因加速频率（boost）策略或芯片分级（binning）而上报不同的 `cpuinfo_max_freq`。因此下面的检查只给出提示，确认平台是否为异构架构还需结合 `lscpu` 输出的核心型号信息。

In [ ]:
import os
import re
import subprocess
import platform

print('=' * 60)
print(' 一、平台信息')
print('=' * 60)
print('操作系统   :', platform.system(), platform.release())
print('处理器架构 :', platform.machine())
print('逻辑核心数 :', os.cpu_count())

print()
print('=' * 60)
print(' 二、编译器与 OpenMP 支持')
print('=' * 60)
gcc_ver = subprocess.run(['gcc', '--version'], capture_output=True,
                         text=True).stdout.splitlines()[0]
print('编译器     :', gcc_ver)

probe = subprocess.run('echo | gcc -fopenmp -dM -E -x c - | grep _OPENMP',
                       shell=True, capture_output=True, text=True).stdout.strip()
if probe:
    ver = int(probe.split()[-1])
    spec = {200805: '3.0', 201107: '3.1', 201307: '4.0',
            201511: '4.5', 201811: '5.0', 202011: '5.1'}.get(ver, '未知')
    print('_OPENMP    :', ver, '(对应 OpenMP %s 规范)' % spec)
    print('数组段归约 :', '支持' if ver >= 201511 else '不支持（需要 4.5 及以上）')
else:
    print('⚠️  未检测到 OpenMP 支持，请确认编译时带有 -fopenmp')

print()
print('=' * 60)
print(' 三、核心频率与异构性检查')
print('=' * 60)
freqs = []
for cpu in range(os.cpu_count() or 1):
    path = '/sys/devices/system/cpu/cpu%d/cpufreq/cpuinfo_max_freq' % cpu
    try:
        with open(path) as f:
            freqs.append((cpu, int(f.read().strip()) // 1000))
    except OSError:
        pass

if not freqs:
    print('无法读取 cpufreq 节点，跳过异构性检查。')
else:
    for cpu, mhz in freqs:
        print('  CPU%-2d 最大频率: %5d MHz' % (cpu, mhz))
    distinct = sorted(set(m for _, m in freqs))
    if len(distinct) > 1:
        print()
        print('⚠️  检测到 %d 种不同的最大频率，本平台可能为异构多核架构（如 Arm big.LITTLE）。'
              % len(distinct))
        print('    请结合 lscpu 输出的核心型号信息进一步确认。')
        print('    若确为异构平台，测速前建议执行：')
        print('      export OMP_PROC_BIND=close')
        print('      export OMP_PLACES=cores')
    else:
        print()
        print('✅ 全部核心的最大频率一致，可按同构多核平台处理。')

print()
print('OMP_NUM_THREADS =', os.environ.get('OMP_NUM_THREADS', '（未设置，由运行时决定）'))
print('OMP_PROC_BIND   =', os.environ.get('OMP_PROC_BIND', '（未设置）'))
print('OMP_PLACES      =', os.environ.get('OMP_PLACES', '（未设置）'))

## 5. 实验工具函数

本节定义三个贯穿全章的辅助函数，后续各实验均直接调用，不再重复说明。

| 函数 | 作用 |
|---|---|
| `compile_c(src)` | 以 `-O3 -fopenmp -Wall -Wextra` 编译指定源文件，并回显全部告警 |
| `run_c(binary, *args)` | 运行可执行文件并原样打印其标准输出 |
| `parse_table(output)` | 从程序输出的结果表中提取「方法名 / 耗时 / 加速比 / 校验」四列 |

**关于编译选项**：全章统一使用 `-O3 -fopenmp`。AArch64 平台的 NEON 属于基线指令集，无需附加 `-march` 或 `-mcpu` 选项。`-Wall -Wextra` 用于暴露数据环境声明不当引发的告警，这类告警在 OpenMP 程序中往往是并发缺陷的征兆，不应忽略。

In [ ]:
import subprocess
import re
import os

SRC_DIR = 'src_histogram'
os.makedirs(SRC_DIR, exist_ok=True)

CFLAGS = ['-O3', '-fopenmp', '-Wall', '-Wextra']


def compile_c(src, extra=('-lm',)):
    """编译单个源文件，返回可执行文件路径；编译失败时抛出异常。"""
    src_path = os.path.join(SRC_DIR, src)
    binary = os.path.join(SRC_DIR, os.path.splitext(src)[0])
    cmd = ['gcc'] + CFLAGS + ['-o', binary, src_path] + list(extra)
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print(proc.stderr.rstrip())
    if proc.returncode != 0:
        raise RuntimeError('编译失败：%s' % src)
    print('✅ 编译通过，无告警' if not proc.stderr.strip()
          else '⚠️  编译通过，但存在告警，请逐条阅读')
    return binary


def run_c(binary, *args, env=None):
    """运行可执行文件，打印并返回其标准输出。"""
    cmd = [binary] + [str(a) for a in args]
    print('$', ' '.join(cmd))
    print()
    run_env = dict(os.environ)
    if env:
        run_env.update({k: str(v) for k, v in env.items()})
    proc = subprocess.run(cmd, capture_output=True, text=True, env=run_env)
    print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print('[stderr]', proc.stderr.rstrip())
    return proc.stdout


ROW_RE = re.compile(r'^\|\s*(.+?)\s*\|\s*([0-9.]+)\s*\|\s*([0-9.]+)x\s*\|\s*(\S+)\s*\|$')


def parse_table(output):
    """解析结果表，返回 [(方法名, 耗时ms, 加速比, 校验结论), ...]。"""
    rows = []
    for line in output.splitlines():
        m = ROW_RE.match(line.strip())
        if m:
            rows.append((m.group(1), float(m.group(2)),
                         float(m.group(3)), m.group(4)))
    return rows


print('工具函数已就绪，源码目录：', os.path.abspath(SRC_DIR))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

C_BASE, C_GOOD, C_FAIL, C_SLOW = '#7f7f7f', '#1f77b4', '#d62728', '#ff7f0e'


def plot_speedup(rows, title, figsize=(10, 5)):
    """绘制加速比柱状图。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。"""
    if not rows:
        print('未解析到结果行，请先运行上一单元格。')
        return
    names = [r[0] for r in rows]
    speeds = [r[2] for r in rows]
    colors = []
    for i, (_, _, sp, chk) in enumerate(rows):
        if i == 0:
            colors.append(C_BASE)
        elif chk == 'FAIL':
            colors.append(C_FAIL)
        elif sp < 1.0:
            colors.append(C_SLOW)
        else:
            colors.append(C_GOOD)

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(range(len(names)), speeds, color=colors,
                  edgecolor='black', linewidth=0.6, width=0.6)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Speedup vs. Serial Baseline')
    ax.set_title(title, fontsize=12, pad=12)
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    ax.set_axisbelow(True)

    for bar, (_, ms, sp, chk) in zip(bars, rows):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                '%.2fx\n%.1f ms%s' % (sp, ms, '' if chk in ('-', 'PASS') else '\nFAIL'),
                ha='center', va='bottom', fontsize=8)

    ax.set_ylim(0, max(speeds) * 1.30)
    plt.tight_layout()
    plt.show()


print('绘图函数已就绪。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。')

## 6. 版本设计总览

| 版本 | 新增计算函数 | 同步方式 | 竞争是否存在 | 结果表行号 |
|---|---|---|---|---|
| **V0** | `hist_serial` | — | — | 1 |
| **V1** | `hist_critical` | 全局临界区 | 管理 | 2 |
| **V2** | `hist_atomic` | 硬件原子指令 | 管理 | 3 |
| **V3** | `hist_locks` | 每 bin 一把锁 | 管理 | 4 |
| **V4** | `hist_local_unpadded` | 无（线程局部） | **消除**，但有伪共享 | 5 |
| **V5** | `hist_local_padded` | 无（线程局部 + 填充） | 消除，无伪共享 | 6 |
| **V6** | `hist_reduction` | 数组段归约 | 由编译器消除 | 7 |

**两条线索**：

- **V1 → V2 → V3**：在「管理竞争」的框架内改进同步手段；
- **V4 → V5 → V6**：跳出该框架，先消除竞争，再解决由此引入的伪共享。

这两条线索的对比本身就是一条重要结论：**消除问题优于管理问题**。

## 7. 逐版本代码讲解

### 7.1 V0 · 串行基准

```c
static void hist_serial(const int *data, long n, int *bins) {
  for (long i = 0; i < n; i++) {
    bins[data[i]]++;
  }
}
```

### 7.2 V1 · 全局临界区

```c
#pragma omp parallel for num_threads(thread_count)
for (long i = 0; i < n; i++) {
  int b = data[i];
#pragma omp critical
  { bins[b]++; }
}
```

注意 `int b = data[i];` 被特意提到临界区**之外**——读取 `data[i]` 不需要互斥，把它放进临界区只会延长串行段。这一写法沿用了实验二 V1 的原则：临界区内只放必须互斥的语句。

即便如此，V1 依然很慢：$n$ 次迭代意味着 $n$ 次加锁解锁，而所有线程共用同一把锁，实际上完全串行执行，还额外付出了锁协议的代价。

### 7.3 V2 · 硬件原子指令

```c
#pragma omp atomic
bins[b]++;
```

`atomic` 按地址互斥，访问不同 bin 的线程互不阻塞。在 bin 数较多时，冲突概率约为 $1/B$，因而 V2 通常比 V1 快数倍。

### 7.4 V3 · 锁数组

```c
omp_set_lock(&locks[b]);
bins[b]++;
omp_unset_lock(&locks[b]);
```

锁的初始化与销毁在计时窗口**之外**完成，因此表格中的耗时只反映加锁解锁本身的代价。

V3 与 V2 的互斥粒度相同（都是按 bin），但 V3 需要两次库函数调用，且锁数组本身也要在核心之间迁移。因此 V3 通常**慢于** V2，有时甚至接近 V1。

> 这是一个值得注意的反例：**更细的锁粒度不一定带来更好的性能**，因为锁本身也是共享数据。

### 7.5 V4 · 线程局部直方图（未填充）

```c
#pragma omp parallel num_threads(thread_count)
{
  int tid = omp_get_thread_num();
  int *my_bins = local_bins + (size_t)tid * (size_t)num_bins;
#pragma omp for
  for (long i = 0; i < n; i++) {
    my_bins[data[i]]++;      // 无任何同步
  }
}
// 串行合并
```

统计阶段完全没有同步，理论上应当接近线性加速。但当 `num_bins` 较小时，多个线程的私有区域会落入同一条缓存行，伪共享随之出现。

**量化判断**：`num_bins = 8` 时每线程占 32 字节，一条 64 字节的缓存行可容纳两个线程的全部 bin，于是每一次自增都可能触发该行在两个核心之间迁移。
`num_bins = 100` 时每线程占 400 字节，跨越约 7 条缓存行，只有边界处的一两条行被共享，影响因而小得多。

### 7.6 V5 · 缓存行填充

```c
size_t padded_bytes =
    ((bytes_per_thread + CACHE_LINE - 1) / CACHE_LINE) * CACHE_LINE;
size_t stride = padded_bytes / sizeof(int);
int *my_bins = local_bins + (size_t)tid * stride;
```

唯一的改动是把索引步长由 `num_bins` 改为 `stride`。程序头部会打印实际的填充结果，便于核对：

```
 Private block: 32 bytes -> padded to 64 bytes (1 lines)
```

### 7.7 V6 · 数组段归约

```c
#pragma omp parallel for num_threads(thread_count) \
    reduction(+ : bins[ : num_bins])
for (long i = 0; i < n; i++) {
  bins[data[i]]++;
}
```

代码回到了与串行版本几乎相同的形态——这正是本章反复出现的主题：**由编译器承担实现细节，程序员只声明意图**。

V6 与 V5 的性能通常接近。若 V6 明显较慢，则可能是该实现未对私有副本做缓存行填充，可通过对比两者加以判断。

## 8. 源代码写入

In [ ]:
%%writefile {SRC_DIR}/omp_histogram.c
#define _POSIX_C_SOURCE 200809L

#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#ifndef _OPENMP
#error "OpenMP is required. Please compile with -fopenmp."
#endif

#if _OPENMP >= 201511
#define HAVE_ARRAY_REDUCTION 1
#else
#define HAVE_ARRAY_REDUCTION 0
#endif

#define NTIMES 5
#define MAX_THREADS 16
#define MAX_BINS 4096
#define CACHE_LINE 64
#define ALIGN_BYTES 64

#define BANNER "============================================================"
#define LINE "------------------------------------------------------------"

// ----------------------------------------------------------------------------
// Common helpers
// ----------------------------------------------------------------------------
static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

static void *alloc_aligned(size_t bytes) {
  size_t rounded = ((bytes + ALIGN_BYTES - 1) / ALIGN_BYTES) * ALIGN_BYTES;
  return aligned_alloc(ALIGN_BYTES, rounded);
}

static int check_equal_int(const int *ref, const int *test, long n) {
  for (long i = 0; i < n; i++) {
    if (ref[i] != test[i]) {
      return 0;
    }
  }
  return 1;
}

static void print_table_header(void) {
  printf("\n%s\n", LINE);
  printf("| %-26s | %9s | %7s | %-5s |\n", "Method", "Time(ms)", "Speedup",
         "Check");
  printf("|----------------------------|-----------|---------|-------|\n");
}

static void print_row(const char *name, double time_ms, double base_ms,
                      int check) {
  const char *status = (check < 0) ? "-" : (check ? "PASS" : "FAIL");
  double speedup = (time_ms > 0.0) ? base_ms / time_ms : 0.0;
  printf("| %-26s | %9.3f | %6.2fx | %-5s |\n", name, time_ms, speedup, status);
}

// ============================================================================
// V0: Serial baseline. The write address is decided by the data itself, which
// is what makes this pattern harder than a plain reduction.
// ============================================================================
static void hist_serial(const int *data, long n, int *bins) {
  for (long i = 0; i < n; i++) {
    bins[data[i]]++;
  }
}

// ============================================================================
// V1: one global critical section per increment.
// ============================================================================
static void hist_critical(const int *data, long n, int thread_count,
                          int *bins) {
#pragma omp parallel for num_threads(thread_count)
  for (long i = 0; i < n; i++) {
    int b = data[i];
#pragma omp critical
    { bins[b]++; }
  }
}

// ============================================================================
// V2: atomic update, backed by a single read-modify-write instruction.
// ============================================================================
static void hist_atomic(const int *data, long n, int thread_count, int *bins) {
#pragma omp parallel for num_threads(thread_count)
  for (long i = 0; i < n; i++) {
    int b = data[i];
#pragma omp atomic
    bins[b]++;
  }
}

// ============================================================================
// V3: one lock per bin. Two threads touching different bins never wait for
// each other, but the lock array itself becomes contended memory.
// ============================================================================
static void hist_locks(const int *data, long n, int thread_count, int *bins,
                       omp_lock_t *locks) {
#pragma omp parallel for num_threads(thread_count)
  for (long i = 0; i < n; i++) {
    int b = data[i];
    omp_set_lock(&locks[b]);
    bins[b]++;
    omp_unset_lock(&locks[b]);
  }
}

// ============================================================================
// V4: private bins per thread, packed back to back. No lock is needed, but the
// last bins of one thread share a cache line with the first bins of the next.
// ============================================================================
static void hist_local_unpadded(const int *data, long n, int num_bins,
                                int thread_count, int *bins) {
  size_t total = (size_t)thread_count * (size_t)num_bins;
  int *local_bins = (int *)alloc_aligned(total * sizeof(int));
  if (local_bins == NULL) {
    return;
  }
  memset(local_bins, 0, total * sizeof(int));

#pragma omp parallel num_threads(thread_count)
  {
    int tid = omp_get_thread_num();
    int *my_bins = local_bins + (size_t)tid * (size_t)num_bins;

#pragma omp for
    for (long i = 0; i < n; i++) {
      my_bins[data[i]]++;
    }
  }

  for (int t = 0; t < thread_count; t++) {
    const int *my_bins = local_bins + (size_t)t * (size_t)num_bins;
    for (int b = 0; b < num_bins; b++) {
      bins[b] += my_bins[b];
    }
  }
  free(local_bins);
}

// ============================================================================
// V5: the same idea, with every private block rounded up to a whole number of
// cache lines so that no line is ever written by two threads.
// ============================================================================
static void hist_local_padded(const int *data, long n, int num_bins,
                              int thread_count, int *bins) {
  size_t bytes_per_thread = (size_t)num_bins * sizeof(int);
  size_t padded_bytes =
      ((bytes_per_thread + CACHE_LINE - 1) / CACHE_LINE) * CACHE_LINE;
  size_t stride = padded_bytes / sizeof(int);

  size_t total = (size_t)thread_count * stride;
  int *local_bins = (int *)alloc_aligned(total * sizeof(int));
  if (local_bins == NULL) {
    return;
  }
  memset(local_bins, 0, total * sizeof(int));

#pragma omp parallel num_threads(thread_count)
  {
    int tid = omp_get_thread_num();
    int *my_bins = local_bins + (size_t)tid * stride;

#pragma omp for
    for (long i = 0; i < n; i++) {
      my_bins[data[i]]++;
    }
  }

  for (int t = 0; t < thread_count; t++) {
    const int *my_bins = local_bins + (size_t)t * stride;
    for (int b = 0; b < num_bins; b++) {
      bins[b] += my_bins[b];
    }
  }
  free(local_bins);
}

#if HAVE_ARRAY_REDUCTION
// ============================================================================
// V6: the compiler creates the private copies and the merge loop by itself.
// ============================================================================
static void hist_reduction(const int *data, long n, int num_bins,
                           int thread_count, int *bins) {
#pragma omp parallel for num_threads(thread_count) \
    reduction(+ : bins[ : num_bins])
  for (long i = 0; i < n; i++) {
    bins[data[i]]++;
  }
}
#endif

int main(int argc, char *argv[]) {
  if (argc != 4) {
    printf("Usage: %s <n_values> <num_bins> <thread_count>\n", argv[0]);
    printf("Example: %s 1000000 100 4\n", argv[0]);
    return 1;
  }

  long n = strtol(argv[1], NULL, 10);
  int num_bins = (int)strtol(argv[2], NULL, 10);
  int thread_count = (int)strtol(argv[3], NULL, 10);

  if (n <= 0) {
    printf("Error: n_values must be > 0\n");
    return 1;
  }
  if (num_bins < 1 || num_bins > MAX_BINS) {
    printf("Error: num_bins must be between 1 and %d\n", MAX_BINS);
    return 1;
  }
  if (thread_count < 1 || thread_count > MAX_THREADS) {
    printf("Error: thread_count must be between 1 and %d\n", MAX_THREADS);
    return 1;
  }

  size_t bytes_per_thread = (size_t)num_bins * sizeof(int);
  size_t padded_bytes =
      ((bytes_per_thread + CACHE_LINE - 1) / CACHE_LINE) * CACHE_LINE;

  printf("%s\n", BANNER);
  printf(" Lab 6: Histogram and False Sharing\n");
  printf(" n: %ld | Bins: %d | Threads: %d | Runs: %d\n", n, num_bins,
         thread_count, NTIMES);
  printf(" Private block: %zu bytes -> padded to %zu bytes (%zu lines)\n",
         bytes_per_thread, padded_bytes, padded_bytes / CACHE_LINE);
  printf(" _OPENMP: %d | Procs: %d\n", _OPENMP, omp_get_num_procs());
  printf("%s\n", BANNER);

  size_t bin_bytes = (size_t)num_bins * sizeof(int);
  int *data = (int *)alloc_aligned((size_t)n * sizeof(int));
  int *b_ref = (int *)alloc_aligned(bin_bytes);
  int *b_test = (int *)alloc_aligned(bin_bytes);
  omp_lock_t *locks =
      (omp_lock_t *)malloc((size_t)num_bins * sizeof(omp_lock_t));

  if (data == NULL || b_ref == NULL || b_test == NULL || locks == NULL) {
    printf("Error: memory allocation failed\n");
    return 1;
  }

  srand(42);
  for (long i = 0; i < n; i++) {
    data[i] = rand() % num_bins;
  }
  for (int b = 0; b < num_bins; b++) {
    omp_init_lock(&locks[b]);
  }

  double start = 0.0;
  double t[7] = {0.0};
  int ok[7] = {0};

  // --- V0 ---
  for (int r = 0; r < NTIMES; r++) {
    memset(b_ref, 0, bin_bytes);
    start = get_time_ms();
    hist_serial(data, n, b_ref);
    t[0] += get_time_ms() - start;
  }
  ok[0] = -1;

  // --- V1 ---
  for (int r = 0; r < NTIMES; r++) {
    memset(b_test, 0, bin_bytes);
    start = get_time_ms();
    hist_critical(data, n, thread_count, b_test);
    t[1] += get_time_ms() - start;
  }
  ok[1] = check_equal_int(b_ref, b_test, num_bins);

  // --- V2 ---
  for (int r = 0; r < NTIMES; r++) {
    memset(b_test, 0, bin_bytes);
    start = get_time_ms();
    hist_atomic(data, n, thread_count, b_test);
    t[2] += get_time_ms() - start;
  }
  ok[2] = check_equal_int(b_ref, b_test, num_bins);

  // --- V3 ---
  for (int r = 0; r < NTIMES; r++) {
    memset(b_test, 0, bin_bytes);
    start = get_time_ms();
    hist_locks(data, n, thread_count, b_test, locks);
    t[3] += get_time_ms() - start;
  }
  ok[3] = check_equal_int(b_ref, b_test, num_bins);

  // --- V4 ---
  for (int r = 0; r < NTIMES; r++) {
    memset(b_test, 0, bin_bytes);
    start = get_time_ms();
    hist_local_unpadded(data, n, num_bins, thread_count, b_test);
    t[4] += get_time_ms() - start;
  }
  ok[4] = check_equal_int(b_ref, b_test, num_bins);

  // --- V5 ---
  for (int r = 0; r < NTIMES; r++) {
    memset(b_test, 0, bin_bytes);
    start = get_time_ms();
    hist_local_padded(data, n, num_bins, thread_count, b_test);
    t[5] += get_time_ms() - start;
  }
  ok[5] = check_equal_int(b_ref, b_test, num_bins);

#if HAVE_ARRAY_REDUCTION
  // --- V6 ---
  for (int r = 0; r < NTIMES; r++) {
    memset(b_test, 0, bin_bytes);
    start = get_time_ms();
    hist_reduction(data, n, num_bins, thread_count, b_test);
    t[6] += get_time_ms() - start;
  }
  ok[6] = check_equal_int(b_ref, b_test, num_bins);
#endif

  for (int i = 0; i < 7; i++) {
    t[i] /= NTIMES;
  }

  print_table_header();
  print_row("V0: Serial Baseline", t[0], t[0], ok[0]);
  print_row("V1: Critical (Global)", t[1], t[0], ok[1]);
  print_row("V2: Atomic", t[2], t[0], ok[2]);
  print_row("V3: Array of Locks", t[3], t[0], ok[3]);
  print_row("V4: Local Bins (Unpadded)", t[4], t[0], ok[4]);
  print_row("V5: Local Bins (Padded)", t[5], t[0], ok[5]);
#if HAVE_ARRAY_REDUCTION
  print_row("V6: Array Reduction", t[6], t[0], ok[6]);
#endif
  printf("%s\n", LINE);

#if !HAVE_ARRAY_REDUCTION
  printf("\nNote: V6 needs OpenMP 4.5 (_OPENMP >= 201511). Current: %d\n",
         _OPENMP);
#endif

  for (int b = 0; b < num_bins; b++) {
    omp_destroy_lock(&locks[b]);
  }
  free(data);
  free(b_ref);
  free(b_test);
  free(locks);
  return 0;
}

## 9. 编译与运行

### 9.1 编译

请留意编译输出：若当前编译器不支持 OpenMP 4.5，`HAVE_ARRAY_REDUCTION` 将为 0，V6 一行会被条件编译排除，程序会在表格之后打印相应说明。

In [ ]:
bin_hist = compile_c('omp_histogram.c', extra=())

### 9.2 第一组：`num_bins = 100`（伪共享影响较小）

每个线程的私有区域为 400 字节，跨越约 7 条缓存行，只有边界处存在共享。此时 V4 与 V5 的差距应当有限。

In [ ]:
out_h100 = run_c(bin_hist, 1000000, 100, 4)
rows_h100 = parse_table(out_h100)

### 9.3 第二组：`num_bins = 8`（伪共享影响显著）

每个线程的私有区域仅 32 字节，两个线程的全部 bin 落在同一条缓存行内。预期 V4 会明显慢于 V5。

同时注意：bin 数减少也会**加剧真实竞争**（冲突概率由 1/100 升至 1/8），因此 V1、V2、V3 三个版本同样会变慢。区分这两种效应正是第 11 节分析的重点。

In [ ]:
out_h8 = run_c(bin_hist, 1000000, 8, 4)
rows_h8 = parse_table(out_h8)

## 10. 结果可视化

### 10.1 两组分别绘图

In [ ]:
plot_speedup(rows_h100,
             'Lab 6: Histogram, num_bins = 100 (n = 1e6, 4 threads)',
             figsize=(11, 5))
plot_speedup(rows_h8,
             'Lab 6: Histogram, num_bins = 8 (n = 1e6, 4 threads)',
             figsize=(11, 5))

### 10.2 两组并列对比

把两组结果画在同一张图上，可以直接看出哪些版本对 bin 数敏感。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

names = [r[0] for r in rows_h100]
sp100 = [r[2] for r in rows_h100]
d8 = {r[0]: r[2] for r in rows_h8}
sp8 = [d8.get(nm, 0.0) for nm in names]

x = np.arange(len(names))
w = 0.38
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w / 2, sp100, w, label='num_bins = 100',
       color='#1f77b4', edgecolor='black', linewidth=0.6)
ax.bar(x + w / 2, sp8, w, label='num_bins = 8',
       color='#ff7f0e', edgecolor='black', linewidth=0.6)
ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Speedup vs. Serial Baseline')
ax.set_title('Lab 6: Effect of Bin Count on Each Strategy')
ax.grid(axis='y', linestyle=':', alpha=0.5)
ax.set_axisbelow(True)
ax.legend()
plt.tight_layout()
plt.show()

### 10.3 伪共享的定量提取

V4 与 V5 的计算量、访存次数、同步次数完全相同，主要差别是私有区域的**步长**（V5 另有少量填充字节的分配与清零开销，相对于 $n$ 次自增可以忽略）。因此两者的耗时之比可作为伪共享代价的近似度量。

In [ ]:
for label, rows in (('num_bins = 100', rows_h100),
                    ('num_bins =   8', rows_h8)):
    d = {r[0]: r[1] for r in rows}
    v4 = d.get('V4: Local Bins (Unpadded)')
    v5 = d.get('V5: Local Bins (Padded)')
    if v4 and v5:
        print('%s   V4 %8.3f ms   V5 %8.3f ms   V4/V5 = %.2f'
              % (label, v4, v5, v4 / v5))
print()
print('该比值可作为伪共享额外代价的度量。bin 数越小，比值应当越大。')
print('在单核环境下缓存行不会在核心之间迁移，两组比值都会接近 1.00。')

## 11. 结果分析

> 以下结论针对**趋势规律**。具体数值随平台、核心数与缓存结构而变化。

**① `critical` 是最慢的方案**

V1 通常比串行基准慢数倍。原因有两层：所有线程共用一把全局锁，使统计阶段实际上串行执行；在此基础上还叠加了 $n$ 次锁协议的开销。**对高频操作使用粗粒度互斥，往往导致严重的性能损失。**

**② `atomic` 明显快于 `critical`，但仍慢于串行**

`atomic` 按地址互斥，冲突概率降为约 $1/B$。但每次自增仍是一次原子操作，其代价远高于普通自增。在采用 LL/SC 实现的平台上（如 AArch64 的 `LDXR` / `STXR`），冲突时还需重试；在提供单条原子加指令的平台上（如 x86 的 `lock add`、ARMv8.1-A LSE 扩展的 `LDADD`）虽无重试，但冲突同样会引起缓存行在核心之间争用。因此 V2 虽比 V1 快数倍，通常仍达不到串行的水平。

**③ 锁数组不如 `atomic`**

V3 的互斥粒度与 V2 相同，却更慢。这说明**锁的粒度并非唯一的考量因素**：锁对象本身的读写代价、函数调用开销，以及锁数组自身的缓存行竞争，均会影响总体性能。

**④ 线程局部化是本实验中能够取得实质性加速的方案**

V4、V5、V6 在统计阶段没有任何同步，因而能够取得接近线性的加速。代价是额外的内存（`thread_count × num_bins` 个计数器）与一次串行合并（$O(p \times B)$，当 $B \ll n$ 时可忽略）。

**⑤ 伪共享的强度取决于私有区域的大小**

这是本实验最重要的观察之一。对比两组结果：

| | `num_bins = 100` | `num_bins = 8` |
|---|---|---|
| 每线程私有区域 | 400 字节 | 32 字节 |
| 跨越缓存行数 | 约 7 条 | 不足 1 条 |
| 共享行的比例 | 边界 1–2 条 | **全部** |
| V4 与 V5 的差距 | 较小 | 显著 |

**判断准则**：当每线程私有区域小于或接近一条缓存行时，伪共享极易发生；当其远大于一条缓存行时，仅边界处受影响，其影响通常可以忽略。

**⑥ 区分两种「bin 数减少带来的变慢」**

bin 数由 100 降到 8 时，所有版本都会变慢，但原因不同：

| 版本 | 变慢的原因 |
|---|---|
| V1 | 无变化（本来就是全局锁，与 bin 数无关） |
| V2、V3 | **真实竞争**加剧，冲突概率由 1/100 升至 1/8 |
| V4 | **伪共享**加剧（逻辑上并无竞争） |
| V5、V6 | 基本无变化（已填充或由实现处理） |

V5 相对于 V4 的稳定性，正是「填充有效」的直接证据。

**⑦ 关于单核环境**

若实验在单核环境下运行，缓存行不会在核心之间迁移，V4 与 V5 的差距会消失。**伪共享是一种多核现象**，与实验三中的数据竞争类似，均需在真实的并发条件下才能观测到。

## 12. 🔧 动手练习

**练习 1**　把 `num_bins` 依次取 2、4、8、16、32、64、128、256，绘制 V4 与 V5 耗时之比随 bin 数变化的曲线，找出伪共享开始显著的临界 bin 数，并与「64 字节 ÷ 4 字节 = 16 个 bin」这一理论值对照。

**练习 2**　把线程数依次取 1、2、4、8，观察 V4 与 V5 的差距如何变化。线程数为 1 时差距应当是多少？为什么？

**练习 3**　把 V5 中的 `CACHE_LINE` 由 64 改为 128 重新编译，观察性能是否进一步提升。再改为 32，观察是否退化。由此推断本机的实际缓存行长度。

**练习 4**　把 `alloc_aligned` 换成普通的 `malloc`，重新运行 V5，观察性能是否发生变化。请解释原因。

**练习 5**（进阶）　用 `objdump -d` 反汇编，对比 V1、V2 两个计算函数所生成的指令序列，找出 `atomic` 对应的 `LDXR` / `STXR` 指令对（x86 平台上为带 `lock` 前缀的指令）。

> 提示：若反汇编中未见 `LDXR` / `STXR`，而是出现了对 `__aarch64_ldadd4_*` 的调用，说明编译器启用了**外联原子**（`-moutline-atomics`，GCC 10 起在 AArch64 上通常默认开启），会在运行时根据 CPU 是否支持 LSE 扩展进行分派。此时可加 `-mno-outline-atomics` 重新编译，即可看到直接生成的 `LDXR` / `STXR` 序列；也可加 `-march=armv8.2-a+lse` 观察 `LDADD` 单指令形式。这三种代码形态的对比本身即有价值。

**练习 6**（进阶）　尝试用 `#pragma omp atomic capture` 改写 V2，同时读出自增前的旧值。这一形式在什么场景下有用？

### 12.1 练习 1 的参考实现：bin 数扫描

In [ ]:
import matplotlib.pyplot as plt

bins_list = [2, 4, 8, 16, 32, 64, 128, 256]
ratio, v4s, v5s = [], [], []
for B in bins_list:
    out = subprocess.run([bin_hist, '1000000', str(B), '4'],
                         capture_output=True, text=True).stdout
    d = {r[0]: r[1] for r in parse_table(out)}
    v4 = d.get('V4: Local Bins (Unpadded)', float('nan'))
    v5 = d.get('V5: Local Bins (Padded)', float('nan'))
    v4s.append(v4)
    v5s.append(v5)
    ratio.append(v4 / v5 if v5 else float('nan'))
    print('bins = %-4d  每线程 %4d 字节   V4 %7.3f ms  V5 %7.3f ms  '
          'V4/V5 = %.2f' % (B, B * 4, v4, v5, ratio[-1]))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(bins_list, ratio, marker='o', color='#d62728')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
ax.axvline(16, color='gray', linestyle=':', linewidth=1.2)
ax.text(16, max(ratio) * 0.95, ' 16 bins = 64 bytes\n = 1 cache line',
        fontsize=9, va='top')
ax.set_xscale('log', base=2)
ax.set_xlabel('num_bins (log2 scale)')
ax.set_ylabel('T(V4 unpadded) / T(V5 padded)')
ax.set_title('Lab 6: False Sharing Penalty vs. Private Block Size')
ax.set_xticks(bins_list)
ax.set_xticklabels([str(b) for b in bins_list])
ax.grid(linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

### 12.2 练习 2 的参考实现：线程数对伪共享的影响

In [ ]:
for nt in (1, 2, 4, 8):
    out = subprocess.run([bin_hist, '1000000', '8', str(nt)],
                         capture_output=True, text=True).stdout
    d = {r[0]: r[1] for r in parse_table(out)}
    v4 = d.get('V4: Local Bins (Unpadded)', float('nan'))
    v5 = d.get('V5: Local Bins (Padded)', float('nan'))
    print('线程数 %-2d   V4 %7.3f ms   V5 %7.3f ms   V4/V5 = %.2f'
          % (nt, v4, v5, v4 / v5 if v5 else float('nan')))
print()
print('线程数为 1 时不存在跨核心的缓存行迁移，比值应当接近 1.00。')

## 13. 🤔 思考题

**思考题 1**　V1 使用无名 `critical`。若改为具名形式 `#pragma omp critical(hist)`，性能会变化吗？在什么样的程序中，具名 `critical` 能带来明显收益？

**思考题 2**　V4 的合并阶段是串行的，复杂度为 $O(p \times B)$。当 $B$ 增大到什么量级时，合并阶段会成为瓶颈？此时应当如何改进？

**思考题 3**　伪共享的三个成立条件中，第一条是「多个线程频繁**写入**」。若多个线程只是频繁**读取**同一条缓存行，会有性能问题吗？请从缓存一致性协议的状态转换角度作答。

**思考题 4**　V5 的填充使内存占用从 $p \times B \times 4$ 字节增加到 $p \times \lceil B/16 \rceil \times 64$ 字节。在 $B = 8$、$p = 8$ 时，这一增幅是多少？是否存在既能消除伪共享又不增加内存的方案？

**思考题 5**　本实验的数据由 `rand() % num_bins` 生成，分布是均匀的。若数据高度倾斜（例如 90% 的元素都落在同一个 bin），V2 与 V4 的表现会各自如何变化？请结合本实验「消除竞争优于管理竞争」这一主线结论作答。

**思考题 6**（综合）　实验二用 `reduction(+ : sum)` 处理标量归约，本实验用 `reduction(+ : bins[:B])` 处理数组归约。两者在实现机制上有何共同点？为什么标量版本不需要考虑伪共享，数组版本却需要？

## 14. 📌 本实验小结

| 概念 | 要点 |
|---|---|
| 不规则归约 | 写入地址由数据决定，无法在编译期确定 |
| `critical` | 全局互斥，高频场景下最慢 |
| `atomic` | 按地址互斥，由硬件读改写指令支撑 |
| 锁数组 | 粒度虽细，但锁本身也是共享数据，通常不如 `atomic` |
| 线程局部化 | 消除竞争而非管理竞争，唯一能真正加速的方案 |
| 伪共享 | 写不同变量，但落在同一条缓存行 |
| 判定条件 | 多线程写 + 地址不同 + 同一缓存行，三者同时成立 |
| 填充 | 私有区域向上取整到整数条缓存行，且起始地址需对齐 |
| 数组段归约 | `reduction(+ : a[:n])`，需 OpenMP 4.5 |

### 一条贯穿全章的原则

> **消除竞争通常优于管理竞争。**

V1 至 V3 的三种同步手段均属于对竞争的**管理**，在本实验的条件下都未能超越串行基准；V4 至 V6 从数据布局入手**消除**竞争，才取得了实质性的加速。

面对竞争密集的并行程序，应优先考虑能否通过数据私有化等方式使线程不必共享，而非首先选择何种同步机制。

### 与后续实验的衔接

至此，本章已经覆盖了并行执行的三类开销：Fork-Join（实验五）、负载不均（实验四）、以及数据竞争与缓存一致性（本实验）。下一个实验转向另一个维度：**线程级并行与指令级并行的协同**——在多核之上，每个核心内部还有向量部件可以利用。